In [3]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH150=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH150_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH150_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]

#Filtriamo i dati

mask = ak.flatten(Fatjet_isMatchedWithA) == 1
x_filtered = x_MH150[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [4]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

x_easy=[x for x in x_plot if 100 < x < 200]

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=150, sigma=5, gamma=1)
m_voigt.limits["mu"]= (125, 175)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 782.7 (χ²/ndof = 17.0)     │              Nfcn = 138              │
│ EDM = 4.56e-08 (Goal: 0.0002)    │            time = 0.2 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   1.099   │   0.006   │            │            │         │         │       │
│ 1 │ mu    │  152.00   │   0.08    │            │            │   125   │   175   │       │
│ 2 │ sigma │   8.18    │   0.22    │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   8.10    │   0.18    │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────┐
│       │      norm        mu     sigma     gamma │
├───────┼─────────────────────────────────────────┤
│  norm │  3.29e-05  0.006e-3 -0.490e-3  0.478e-3 │
│    mu │  0.006e-3   0.00611    -0.003     0.000 │
│ sigma │ -0.490e-3    -0.003    0.0467    -0.034 │
│ gamma │  0.478e-3     0.000    -0.034    0.0325 │
└───────┴─────────────────────────────────────────┘

In [5]:
fit_MH150_values={}
fit_MH150_errors={}

fit_values={'MH150': fit_MH150_values,}
fit_errors={'MH150_errors': fit_MH150_errors}

for param in m_voigt.parameters:
    fit_MH150_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH150_errors[error] = m_voigt.errors[error]

print(fit_MH150_values)
print(fit_MH150_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH150"]=fit_MH150_values
results["MH150_errors"]=fit_MH150_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH150"]=fit_MH150_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH150_errors"]=fit_MH150_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  


{'norm': 1.0994792867024432, 'mu': 152.00046961694704, 'sigma': 8.180081551567715, 'gamma': 8.100519694549014}
{'norm': 0.005737903404613004, 'mu': 0.07816990769154586, 'sigma': 0.21615087449181702, 'gamma': 0.1802325587054443}
